In [2]:
pip install easyocr

Note: you may need to restart the kernel to use updated packages.


In [19]:
import easyocr
reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [26]:
result = reader.readtext('samplemenu.jpg')
print(result)

c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[([[np.int32(376), np.int32(240)], [np.int32(762), np.int32(240)], [np.int32(762), np.int32(388)], [np.int32(376), np.int32(388)]], 'MENU', np.float64(0.9999492168426514)), ([[np.int32(439), np.int32(461)], [np.int32(697), np.int32(461)], [np.int32(697), np.int32(501)], [np.int32(439), np.int32(501)]], 'APPETIZER', np.float64(0.8699695641258216)), ([[np.int32(410), np.int32(518)], [np.int32(494), np.int32(518)], [np.int32(494), np.int32(542)], [np.int32(410), np.int32(542)]], 'PRAWN', np.float64(0.9997200539837217)), ([[np.int32(502), np.int32(517)], [np.int32(720), np.int32(517)], [np.int32(720), np.int32(542)], [np.int32(502), np.int32(542)]], '& AVOCADO SALAD', np.float64(0.9719048788217238)), ([[np.int32(326), np.int32(562)], [np.int32(648), np.int32(562)], [np.int32(648), np.int32(588)], [np.int32(326), np.int32(588)]], 'LOCAL ROCK OYSTERS WITH', np.float64(0.8807727967396987)), ([[np.int32(654), np.int32(562)], [np.int32(806), np.int32(562)], [np.int32(806), np.int32(586)], [np.i

In [27]:
def to_rect(box):
    xlist = [p[0] for p in box] #0 = x coordinates
    ylist = [p[1] for p in box] #1 = y coordinates
    return {
        "left": min(xlist),
        "right": max(xlist),
        "top": min(ylist),
        "bottom": max(ylist),
        "height": max(ylist) - min(ylist),
        "width": max(xlist) - min(xlist),
        "center_y": (min(ylist) + max(ylist)) / 2
    }


def same_line(rect_a, rect_b):
    #if center y is different by 1 pixel, same line. 
    if abs(rect_a["center_y"] - rect_b["center_y"]) > 1:
        return False
    
    gap = max(0, min(rect_b["left"], rect_a["left"]) - max(rect_a["right"], rect_b["right"]))
    if gap > 10:
        return False
    
    return True

    

In [28]:

structured = []
for bbox, text, conf in result:
    structured.append({
        "rect": to_rect(bbox),
        "text": text,
        "conf": conf
    })

print(structured)    

[{'rect': {'left': np.int32(376), 'right': np.int32(762), 'top': np.int32(240), 'bottom': np.int32(388), 'height': np.int32(148), 'width': np.int32(386), 'center_y': np.float64(314.0)}, 'text': 'MENU', 'conf': np.float64(0.9999492168426514)}, {'rect': {'left': np.int32(439), 'right': np.int32(697), 'top': np.int32(461), 'bottom': np.int32(501), 'height': np.int32(40), 'width': np.int32(258), 'center_y': np.float64(481.0)}, 'text': 'APPETIZER', 'conf': np.float64(0.8699695641258216)}, {'rect': {'left': np.int32(410), 'right': np.int32(494), 'top': np.int32(518), 'bottom': np.int32(542), 'height': np.int32(24), 'width': np.int32(84), 'center_y': np.float64(530.0)}, 'text': 'PRAWN', 'conf': np.float64(0.9997200539837217)}, {'rect': {'left': np.int32(502), 'right': np.int32(720), 'top': np.int32(517), 'bottom': np.int32(542), 'height': np.int32(25), 'width': np.int32(218), 'center_y': np.float64(529.5)}, 'text': '& AVOCADO SALAD', 'conf': np.float64(0.9719048788217238)}, {'rect': {'left': 

In [29]:
lines = []

for obj in structured:
    placed = False
    for line in lines:
        if same_line(obj["rect"], line[-1]["rect"]):
            line.append(obj)
            placed = True
            break
    if not placed:
        lines.append([obj])


In [32]:
lines_text = []

for line in lines:
    text = ""
    for part in line:
        text = text + part["text"] + " "
    lines_text.append(text)

print(lines_text)

['MENU ', 'APPETIZER ', 'PRAWN & AVOCADO SALAD ', 'LOCAL ROCK OYSTERS WITH VINAIGRETTE ', 'PUMPKIN, SAGE & PARMESAN RAVIOLI ', 'ENTREE ', 'BEEF CARPACCIO WITH CAPERS IN OLIVE OIL ', 'RAW KINGFISH TARTARE & EDAMAME PUREE ', 'PRosCiutto MICRO HERB SALAD ', 'MAIN ', 'AGED EYE FILLET STEAK WITH FRIES ', 'CRISPY SKIN SALMON FILLET WITH GREEN BEANS ', 'SPAGHETTI BOLOGNESE WITH PARMESAN ', 'DES SERT ', 'CITRUS & THYME TART WITH DOUBLE CREAM ', 'CARAMEL APPLE PIE WITH VANILLA BEAN ICE CREAM ', 'ICE CREAM CHOCOLATE, VANILLA & STRAWBERRY ']
